In [11]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
@author: Jordy Thielen (jordy.thielen@donders.ru.nl)
"""

import os
from os.path import join
import numpy as np
import pyntbci

wd = r'C:\Users\Radovan\OneDrive\Radboud\Studentships\Jordy Thielen\root'
os.chdir(wd)
data_dir = join(wd, "data")
experiment_dir = join(data_dir, "experiment")
files_dir = join(experiment_dir, 'files')
sourcedata_dir = join(experiment_dir, 'sourcedata')
derivatives_dir = join(join(experiment_dir, 'derivatives'))
os.chdir(wd)
data_dir = experiment_dir

subjects = [
    "VPpdia", "VPpdib", "VPpdic", "VPpdid", "VPpdie", "VPpdif", "VPpdig", "VPpdih", "VPpdii", "VPpdij", "VPpdik",
    "VPpdil", "VPpdim", "VPpdin", "VPpdio", "VPpdip", "VPpdiq", "VPpdir", "VPpdis", "VPpdit", "VPpdiu", "VPpdiv",
    "VPpdiw", "VPpdix", "VPpdiy", "VPpdiz", "VPpdiza", "VPpdizb", "VPpdizc"
]
tasks = ["overt", "covert"]

event = "dur"
onset_event = True
encoding_length = 0.3
ensemble = True
n_folds = 4

# Loop participants
accuracy = np.zeros((len(subjects), len(tasks), n_folds))
for i_subject, subject in enumerate(subjects):
    print(f"{subject}", end="\t")

    # Loop tasks
    for i_task, task in enumerate(tasks):
        print(f"{task}: ", end="")

        # Load data
        file_dir = os.path.join(derivatives_dir, 'preprocessed', "cvep", f"sub-{subject}")
        fn = os.path.join(file_dir, f"sub-{subject}_task-{task}_cvep_64_icaspace.npz")        
        tmp = np.load(fn)
        fs = int(tmp["fs"])
        X = tmp["X"]
        y = tmp["y"]
        V = tmp["V"]

        # Cross-validation
        folds = np.repeat(np.arange(n_folds), int(X.shape[0] / n_folds))
        for i_fold in range(n_folds):
            # Split data to train and test set
            X_trn, y_trn = X[folds != i_fold, :, :], y[folds != i_fold]
            X_tst, y_tst = X[folds == i_fold, :, :], y[folds == i_fold]

            # Train classifier
            rcca = pyntbci.classifiers.rCCA(stimulus=V, fs=fs, event=event, encoding_length=encoding_length,
                                            onset_event=onset_event, ensemble=ensemble)
            rcca.fit(X_trn, y_trn)

            # Apply classifier
            yh_tst = rcca.predict(X_tst)

            # Compute accuracy
            accuracy[i_subject, i_task, i_fold] = np.mean(yh_tst == y_tst)

        print(f"{accuracy[i_subject, i_task, :].mean():.3f}", end="\t")
    print()

print(f"Average:\tovert: {accuracy[:, 0, :].mean():.3f}\tcovert: {accuracy[:, 1, :].mean():.3f}")

#np.savez(os.path.join(data_dir, "derivatives", "cvep_rcca.npz"), accuracy=accuracy)


VPpdia	overt: 1.000	covert: 0.600	
VPpdib	overt: 1.000	covert: 0.600	
VPpdic	overt: 1.000	covert: 0.588	
VPpdid	overt: 1.000	covert: 0.625	
VPpdie	overt: 1.000	covert: 0.625	
VPpdif	overt: 1.000	covert: 0.700	
VPpdig	overt: 1.000	covert: 0.738	
VPpdih	overt: 1.000	covert: 0.738	
VPpdii	overt: 1.000	covert: 0.637	
VPpdij	overt: 1.000	covert: 0.650	
VPpdik	overt: 1.000	covert: 0.775	
VPpdil	overt: 1.000	covert: 0.588	
VPpdim	overt: 1.000	covert: 0.625	
VPpdin	overt: 0.950	covert: 0.850	
VPpdio	overt: 1.000	covert: 0.675	
VPpdip	overt: 1.000	covert: 0.875	
VPpdiq	overt: 1.000	covert: 0.600	
VPpdir	overt: 1.000	covert: 0.812	
VPpdis	overt: 0.950	covert: 0.575	
VPpdit	overt: 1.000	covert: 0.775	
VPpdiu	overt: 1.000	covert: 0.750	
VPpdiv	overt: 1.000	covert: 0.800	
VPpdiw	overt: 0.950	covert: 0.562	
VPpdix	overt: 1.000	covert: 0.662	
VPpdiy	overt: 1.000	covert: 0.562	
VPpdiz	overt: 1.000	covert: 0.600	
VPpdiza	overt: 1.000	covert: 0.588	
VPpdizb	overt: 1.000	covert: 0.587	
VPpdizc	overt: 1.0

In [12]:
manuscript_dir = r'C:\Users\Radovan\OneDrive\Radboud\Studentships\Jordy Thielen\Manuscript'
m_ica_dir = join(manuscript_dir, 'data', 'anova', 'ica')
m_noica_dir = join(manuscript_dir,'data', 'anova', 'noica')

save_path = os.path.join(m_ica_dir, "covert_lda_rcca_dec_64_ica.npz")

# Covert acc
acc = accuracy.mean(axis=1)

n_folds = acc.shape[1]
se = acc.std(axis=1, ddof=1) / np.sqrt(n_folds)

np.savez(
    save_path,
    subject=subjects,
    accuracies=acc,
    ses=se
)

print(f"results saved to {save_path}")

results saved to C:\Users\Radovan\OneDrive\Radboud\Studentships\Jordy Thielen\Manuscript\data\anova\ica\covert_lda_rcca_dec_64_ica.npz
